# GraphAgents: Knowledge Graph-Guided Agentic AI for Cross-Domain Materials Design

#### Authors: Isabella Stewart, Tarjei Hage, Yu-Chuan (Michael) Hsu, and Markus J. Buehler, MIT, 2025 
#### Corresponding author: Markus J. Buehler, mbuehler@MIT.EDU
#### LAMM, Massachusetts Institute of Technology

In [2]:
import os
from GraphReasoning import *


In [3]:
verbatim=False

In [22]:
doc_data_dir = '/home/mkychsu/pool/SG_materialproperties/'
data_dir='./GRAPHDATA'    
data_dir_output='./GRAPHDATA_OUTPUT'
n_ctx = 20000

embedding_file=f'{data_dir_output}/SG_LLAMA33_70b.pkl'


### Load dataset

In [5]:
import pandas as pd
import glob
try:
    df = pd.read_csv(f'{doc_data_dir}/materialproperties.csv', index_col=0)
    
except:
    
    doc_list=sorted(glob.glob(f'{doc_data_dir}/*.xls'))
    df_list = []
    for i, doc in enumerate(doc_list):
        print(i, doc)
        df_list.append(pd.read_excel(doc))
        
    df = pd.concat(df_list, axis=0)
    df = df.drop_duplicates()
    df = df.reset_index(drop=True)
    df.to_csv(f'{doc_data_dir}/materialproperties.csv', drop_index=True)


In [8]:
# # In[6]:


from transformers import AutoModelForCausalLM, AutoTokenizer
# from tqdm.notebook import tqdm
# from IPython.display import display, Markdown


# tokenizer_model=f'/home/mkychsu/pool/llm/SEMIKONG-8b-GPTQ'
tokenizer_model=f'/home/mkychsu/pool/llm/nomic-embed-text-v1.5'

# embedding_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model,use_fast=False, device_map="cuda:0")
# embedding_model = AutoModelForCausalLM.from_pretrained(tokenizer_model,output_hidden_states=True).to('cuda:0')

from sentence_transformers import SentenceTransformer
embedding_tokenizer =''
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)


from GraphReasoning import load_embeddings, save_embeddings, generate_node_embeddings

# generate_new_embeddings=True

# from PIL import Image
# from transformers import AutoModelForCausalLM 
from transformers import AutoProcessor 

model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True, torch_dtype="auto")
processor = AutoProcessor.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True) 




/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<All keys matched successfully>
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.30s/it]
/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/transformers/models/auto/image_processing_auto.py:590: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


In [9]:
import torch

if os.path.exists(f'{data_dir}/{embedding_file}'):
    generate_new_embeddings=False

generate_new_embeddings=True

with torch.no_grad():
    if generate_new_embeddings:

        try:
            import networkx as nx

            graph_root=embedding_file.split('.')[0]
            graph_GraphML= f'{data_dir_output}/{graph_root}.graphml'
            G = nx.read_graphml(graph_GraphML)
            node_embeddings = generate_node_embeddings(G, embedding_tokenizer, embedding_model, )
        except:
            node_embeddings = generate_node_embeddings(nx.DiGraph(), embedding_tokenizer, embedding_model, )

        save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')

    else:
        filename = f"{data_dir}/{embedding_file}"
        node_embeddings = load_embeddings(f'{data_dir}/{embedding_file}')
        
        


0it [00:00, ?it/s]


### Set up LLM client:

In [10]:
# from llama_cpp import Llama

# from llama_cpp.llama_speculative import LlamaPromptLookupDecoding

# llm = Llama(model_path=file_path,
#              n_gpu_layers=-1,verbose= True, #False,#False,
#              n_ctx=n_ctx,
#              main_gpu=0,
#              n_threads= 4,
#              n_threads_batch=32,
#              draft_model=LlamaPromptLookupDecoding(num_pred_tokens=2),
#              logits_all=True,
#              # chat_format='mistral-instruct',
#              )
# # In[10]:

In [11]:

import instructor
from typing import List
from PIL import Image

from pydantic import BaseModel

class Node(BaseModel):
    id: str
    type: str
        
class Edge(BaseModel):
    source: str
    target: str
    relation: str
        
class KnowledgeGraph(BaseModel):
    nodes: List[Node]
    edges: List[Edge]

response_model = KnowledgeGraph
system_prompt = '''
You are a scientific assistant extracting knowledge graphs from text.
Return a JSON with two fields: <nodes> and <edges>.\n
Each node must have <id> and <type>.\n
Each edge must have <source>, <target>, and <relation>.
'''

def generate(system_prompt=system_prompt, 
             prompt="",temperature=0.333,
             max_tokens=n_ctx, response_model=KnowledgeGraph, 
            ):     

    if system_prompt==None:
        messages=[
            {"role": "user", "content": f"{prompt}"},
        ]

    else:
        messages=[
            {"role": "system",  "content": f"{system_prompt}"},
            {"role": "user", "content": f"{prompt}"},
        ]

    if 'json' in prompt.lower() and 'graph' in prompt.lower():
        create = instructor.patch(
            create=llm.create_chat_completion_openai_v1,
            mode=instructor.Mode.JSON_SCHEMA,
        )

        result = create(messages=messages, 
                        temperature=temperature,
                        max_tokens=max_tokens,
                        response_model=response_model,
                       )
        return result
    else:
        
        result=llm.create_chat_completion_openai_v1(
    
        # result=llm.create_chat_completion(
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
        return result.choices[0].message.content #['choices'][0]['message']['content']



def generate_figure(image, system_prompt=system_prompt, 
                prompt="", model=model, processor=processor, temperature=0,
                           ):
    if system_prompt==None:
        messages=[
            {"role": "user", "content": f"Here is the image: <|image_1|>.\n" + prompt},
        ]

    else:
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user", "content": f"Here is the image: <|image_1|>.\n" + prompt},
        ]
        
    try:
        pwd = os.getcwd()
        image = image.split(pwd)[-1]
        image=Path('.').glob(f'**/{image}', case_sensitive=False)
        image = list(image)[0]
    except:
        return '' 
    image = Image.open(image)
    print(f'Extracting infomation from {image}')
    prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors="pt").to("cuda:1") 
    generation_args = { 
                        "max_new_tokens": 1024, 
                        "temperature": 0.1, 
                        "do_sample": True, 
                        "stop_strings": ['<|end|>',
                                         '<|endoftext|>'],
                        "tokenizer": processor.tokenizer,
                      } 

    generate_ids = model.generate(**inputs, eos_token_id=processor.tokenizer.eos_token_id, **generation_args) 

    # remove input tokens 
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0] 
    

In [ ]:
import networkx as nx
from GraphReasoning import make_graph_from_text, add_new_subgraph_from_text, save_embeddings
G=nx.DiGraph()
with torch.no_grad():
    # for i, doc in enumerate(doc_list):
    for i, row in df.iterrows():

        title = row['Article Title']
        title=title.replace('/','|')
        doi = row['DOI']
        graph_root = f'{title}'

        _graph_GraphML= f'{data_dir_output}/{graph_root[:min(200, len(graph_root)-1)]}_augmented_graphML_integrated.graphml'
        txt=row['Abstract']
        # print(f'{doc}')
        image_list = ''#glob.glob('/'.join(doc.split('/')[:-1])+'/*png')
        # break

        current_graph = f'{data_dir}/{graph_root}_graph.graphml'
        if os.path.exists(_graph_GraphML):
            print(f'Main KG will be loaded: {_graph_GraphML}')
            KG_to_load = _graph_GraphML
            continue
        else:
            KG_to_load = ''
            
        if os.path.exists(f'{title}_err.txt'):
            print(f'No. {i}: {title} got something wrong.')
            continue

        if not os.path.exists(current_graph):
            if str(txt)=='nan':
                print(f'No. {i}: content nan')
            
                continue
            else:
                print(f'No. {i}: for {current_graph} as it is not ready yet')
                # break
                now = datetime.now()
                _, current_graph, _, _, _ = make_graph_from_text(txt,generate,
                                      generate_figure, image_list,
                                      graph_root=graph_root[:min(200, len(graph_root)-1)],do_distill=False,
                                      chunk_size=200000,chunk_overlap=0,
                                      repeat_refine=0,verbatim=False,
                                      data_dir=data_dir,
                                                                 
                                      save_PDF=False,
                                     )
                print("Time: ", datetime.now()-now)

        else:
            if KG_to_load:
                G = nx.read_graphml(KG_to_load)
                print(G)
                
            # now = datetime.now()
            
            # print(f'Merging graph No. {i}: {title} to the main one')
            # _, G, _, node_embeddings, _ = add_new_subgraph_from_text(txt='',
            #                    node_embeddings=node_embeddings,
            #                    tokenizer=embedding_tokenizer,
            #                    model=embedding_model,
            #                    original_graph=G, data_dir_output=data_dir_output, graph_root=graph_root[:min(200, len(graph_root)-1)],
            #                    do_simplify_graph=True,size_threshold=0,
            #                    repeat_refine=0,similarity_threshold=0.97,
            #                    do_Louvain_on_new_graph=True,
            #                    #whether or not to simplify, uses similiraty_threshold defined above
            #                    return_only_giant_component=False,
            #                    save_common_graph=False,G_to_add=None,graph_GraphML_to_add=current_graph,
            #                    verbatim=True,)
            # save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')
            # print("Time: ", datetime.now()-now)
            

       

Main KG will be loaded: ./GRAPHDATA_OUTPUT/Designing the new generation of intelligent biocompatible carriers for protein and peptide deliver_augmented_graphML_integrated.graphml
Main KG will be loaded: ./GRAPHDATA_OUTPUT/A long-acting formulation of rifabutin is effective for prevention and treatment of Mycobacterium tuberculosi_augmented_graphML_integrated.graphml
Main KG will be loaded: ./GRAPHDATA_OUTPUT/A Rapid Prototyping Strategy for Manufacturing of Personalized Bolu_augmented_graphML_integrated.graphml
Main KG will be loaded: ./GRAPHDATA_OUTPUT/Long Acting Polycaprolactone Based Parenteral Formulation of Aripiprazole Targeting Behavioural and Biochemical Deficit in Schizophreni_augmented_graphML_integrated.graphml
Main KG will be loaded: ./GRAPHDATA_OUTPUT/A New Level A Type IVIVC for the Rational Design of Clinical Trials Toward Regulatory Approval of Generic Polymeric Long-Acting Injectable_augmented_graphML_integrated.graphml
Main KG will be loaded: ./GRAPHDATA_OUTPUT/GROWT

### Test

In [13]:
title = './GRAPHDATA_OUTPUT/Confined crystallization phenomena in immiscible polymer blends with dispersed micro-and nanometer sized PA6 droplets part 4: polymorphous structure and (meta)-stability of PA6 crystals formed in different temperature regions_augmented_graphML_integrated.graphml'
print(title)

./GRAPHDATA_OUTPUT/Confined crystallization phenomena in immiscible polymer blends with dispersed micro-and nanometer sized PA6 droplets part 4: polymorphous structure and (meta)-stability of PA6 crystals formed in different temperature regions_augmented_graphML_integrated.graphml
